# Lakebase Sync Validation — Read Path

## Architecture

**Gold MV in UC** → **SNAPSHOT synced table** → **Lakebase `fe_bar_db`, schema `fraud`**

| Layer | Location | Tables |
| --- | --- | --- |
| Gold (source) | `workshop_guy_catalog.fe_bar_a0` | `txn_features_gold` (PK `txn_id`, 25 cols), `customer_risk_profile_gold` (PK `customer_id`, 12 cols) |
| Lakebase (target) | `fe_bar_lakebase.fraud` / Postgres `fe_bar_db.fraud` | `txn_features`, `customer_risk_profile` |

## Why SNAPSHOT sync mode

The gold materialized views are **fully recomputed** on every pipeline refresh. A full snapshot copy is therefore equivalent to an incremental sync and avoids the **Beta automatic-CDF dependency** (`MATERIALIZED_VIEW_CDF_REQUIRES_EXTERNAL_METADATA`). Triggered sync was attempted first but failed because MV change data feed requires the Beta `pipelines.externalMetadata.enabled` feature on the PREVIEW channel.

## Lakebase project

* **Project:** `fe-bar-fraud-analytics`
* **Branch:** `production`
* **Endpoint:** `primary` (1 CU, scale-to-zero 300s)
* **Database:** `fe_bar_db`
* **UC Catalog:** `fe_bar_lakebase` (MANAGED_ONLINE_CATALOG)
* **Sync pipeline:** `f8690359-4bb1-45b9-8fdb-5bd892239db2`

In [0]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "psycopg[binary]>=3.1.0", "databricks-sdk>=0.118.0"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import psycopg, time, statistics
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
branch = "projects/fe-bar-fraud-analytics/branches/production"
ep = w.postgres.get_endpoint(name=f"{branch}/endpoints/primary")
HOST = ep.status.hosts.host
USERNAME = w.current_user.me().user_name
TOKEN = w.postgres.generate_database_credential(endpoint=ep.name).token
DBNAME = "fe_bar_db"

def get_pg_conn(read_only=True):
    conn = psycopg.connect(host=HOST, dbname=DBNAME, user=USERNAME, password=TOKEN, sslmode="require")
    conn.read_only = read_only
    return conn

print(f"Host: {HOST}")
print(f"User: {USERNAME}")
print(f"Database: {DBNAME}")
print("Connection ready.")

Host: ep-aged-sky-d2afekn0.database.us-east-1.cloud.databricks.com
User: alex.owen@databricks.com
Database: fe_bar_db
Connection ready.


In [0]:
with get_pg_conn() as conn:
    with conn.cursor() as cur:
        for table in ["fraud.txn_features", "fraud.customer_risk_profile"]:
            schema, tbl = table.split(".")
            cur.execute("""
                SELECT column_name, data_type, is_nullable
                FROM information_schema.columns
                WHERE table_schema = %s AND table_name = %s
                ORDER BY ordinal_position
            """, (schema, tbl))
            cols = cur.fetchall()
            print(f"\n=== {table} ({len(cols)} columns) ===")
            for name, dtype, nullable in cols:
                print(f"  {name:35s} {dtype:30s} nullable={nullable}")

        # Confirm the 3 added columns are present in txn_features
        cur.execute("""
            SELECT column_name FROM information_schema.columns
            WHERE table_schema = 'fraud' AND table_name = 'txn_features'
              AND column_name IN ('geo_distance_from_home', 'device_id', 'channel')
        """)
        found = sorted([r[0] for r in cur.fetchall()])
        print(f"\n✓ Confirmed present in fraud.txn_features: {found}")
        assert found == ['channel', 'device_id', 'geo_distance_from_home'], f"Missing columns: {found}"


=== fraud.txn_features (25 columns) ===
  txn_id                              text                           nullable=NO
  customer_id                         text                           nullable=YES
  merchant_id                         text                           nullable=YES
  txn_ts                              timestamp with time zone       nullable=YES
  amount                              double precision               nullable=YES
  amt_to_monthly_spend                double precision               nullable=YES
  home_country                        text                           nullable=YES
  is_cross_border                     integer                        nullable=YES
  geo_distance_from_home              double precision               nullable=YES
  is_new_device                       integer                        nullable=YES
  device_id                           text                           nullable=YES
  is_online                           integer             

In [0]:
# UC counts
uc_txn = spark.sql("SELECT COUNT(*) as c FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold").collect()[0].c
uc_cust = spark.sql("SELECT COUNT(*) as c FROM workshop_guy_catalog.fe_bar_a0.customer_risk_profile_gold").collect()[0].c

# Postgres counts
with get_pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM fraud.txn_features")
        pg_txn = cur.fetchone()[0]
        cur.execute("SELECT COUNT(*) FROM fraud.customer_risk_profile")
        pg_cust = cur.fetchone()[0]

print(f"{'Table':<45} {'UC':>10} {'Postgres':>10} {'Match':>6}")
print("-" * 75)
print(f"{'txn_features_gold → fraud.txn_features':<45} {uc_txn:>10,} {pg_txn:>10,} {'✓' if uc_txn == pg_txn else '✗':>6}")
print(f"{'customer_risk_profile_gold → fraud.customer_risk_profile':<45} {uc_cust:>10,} {pg_cust:>10,} {'✓' if uc_cust == pg_cust else '✗':>6}")
assert uc_txn == pg_txn and uc_cust == pg_cust, "Row count mismatch!"

Table                                                 UC   Postgres  Match
---------------------------------------------------------------------------
txn_features_gold → fraud.txn_features           464,632    464,632      ✓
customer_risk_profile_gold → fraud.customer_risk_profile      9,289      9,289      ✓


In [0]:
import random

# Get 5 random txn_ids
with get_pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT txn_id FROM fraud.txn_features ORDER BY random() LIMIT 5")
        sample_txns = [r[0] for r in cur.fetchall()]
        cur.execute("SELECT customer_id FROM fraud.customer_risk_profile ORDER BY random() LIMIT 5")
        sample_custs = [r[0] for r in cur.fetchall()]

print("=== Transaction spot check ===")
for txn_id in sample_txns:
    uc_row = spark.sql(f"SELECT txn_id, customer_id, amount, channel, is_cross_border FROM workshop_guy_catalog.fe_bar_a0.txn_features_gold WHERE txn_id = '{txn_id}'").collect()[0]
    with get_pg_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT txn_id, customer_id, amount, channel, is_cross_border FROM fraud.txn_features WHERE txn_id = %s", (txn_id,))
            pg_row = cur.fetchone()
    match = (str(uc_row.txn_id) == str(pg_row[0]) and str(uc_row.customer_id) == str(pg_row[1]) and
             abs(float(uc_row.amount) - float(pg_row[2])) < 0.01)
    print(f"  {txn_id}: UC=({uc_row.customer_id}, {uc_row.amount}, {uc_row.channel}) PG=({pg_row[1]}, {pg_row[2]}, {pg_row[3]}) match={match}")

print("\n=== Customer spot check ===")
for cust_id in sample_custs:
    uc_row = spark.sql(f"SELECT customer_id, tenure, home_region, risk_segment FROM workshop_guy_catalog.fe_bar_a0.customer_risk_profile_gold WHERE customer_id = '{cust_id}'").collect()[0]
    with get_pg_conn() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT customer_id, tenure, home_region, risk_segment FROM fraud.customer_risk_profile WHERE customer_id = %s", (cust_id,))
            pg_row = cur.fetchone()
    match = (str(uc_row.customer_id) == str(pg_row[0]) and int(uc_row.tenure) == int(pg_row[1]))
    print(f"  {cust_id}: UC=({uc_row.tenure}, {uc_row.home_region}, {uc_row.risk_segment}) PG=({pg_row[1]}, {pg_row[2]}, {pg_row[3]}) match={match}")

=== Transaction spot check ===
  TXN-0336248: UC=(CUST-04103, 63.69, online) PG=(CUST-04103, 63.69, online) match=True
  TXN-0436980: UC=(CUST-00683, 58.59, online) PG=(CUST-00683, 58.59, online) match=True
  TXN-0356360: UC=(CUST-06471, 39.31, card_present) PG=(CUST-06471, 39.31, card_present) match=True
  TXN-0053821: UC=(CUST-09431, 761.1, online) PG=(CUST-09431, 761.1, online) match=True
  TXN-0065825: UC=(CUST-08695, 117.12, card_present) PG=(CUST-08695, 117.12, card_present) match=True

=== Customer spot check ===
  CUST-09913: UC=(54, US-East, low) PG=(54, US-East, low) match=True
  CUST-01673: UC=(38, US-East, low) PG=(38, US-East, low) match=True
  CUST-02393: UC=(52, US-West, medium) PG=(52, US-West, medium) match=True
  CUST-05963: UC=(119, EU, low) PG=(119, EU, low) match=True
  CUST-00430: UC=(32, US-East, medium) PG=(32, US-East, medium) match=True


In [0]:
with get_pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT txn_id, customer_id, amount, amt_to_monthly_spend, is_cross_border,
                   geo_distance_from_home, device_id, channel
            FROM fraud.txn_features
            WHERE is_cross_border = 1 AND amt_to_monthly_spend IS NOT NULL
            ORDER BY amt_to_monthly_spend DESC
            LIMIT 5
        """)
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()

print(f"Top 5 cross-border transactions by amt_to_monthly_spend (from Postgres):")
print(f"{'txn_id':<16} {'customer_id':<12} {'amount':>10} {'amt/spend':>10} {'geo_dist':>10} {'device_id':<20} {'channel':<14}")
print("-" * 100)
for r in rows:
    print(f"{r[0]:<16} {r[1]:<12} {r[2]:>10.2f} {r[3]:>10.2f} {r[5]:>10.2f} {r[6]:<20} {r[7]:<14}")

Top 5 cross-border transactions by amt_to_monthly_spend (from Postgres):
txn_id           customer_id      amount  amt/spend   geo_dist device_id            channel       
----------------------------------------------------------------------------------------------------
TXN-0142590      CUST-04866     10877.10      87.78     597.56 DEV-NNWYU-048652     card_present  
TXN-0024013      CUST-05184      8882.33      71.34     516.39 DEV-CATJH-051833     card_present  
TXN-0056358      CUST-03878      5274.84      50.81     730.64 DEV-HMXQQ-038776     card_present  
TXN-0438621      CUST-09294      6878.07      50.43     679.51 DEV-SEXCR-092934     online        
TXN-0090132      CUST-06567      7237.18      49.88     622.89 DEV-RIFTZ-065663     online        


In [0]:
# Get 200 random txn_ids for the benchmark
with get_pg_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT txn_id FROM fraud.txn_features ORDER BY random() LIMIT 201")
        benchmark_ids = [r[0] for r in cur.fetchall()]

warmup_id = benchmark_ids[0]
lookup_ids = benchmark_ids[1:201]

conn = get_pg_conn()
cur = conn.cursor()

# Warm-up query
cur.execute("SELECT * FROM fraud.txn_features WHERE txn_id = %s", (warmup_id,))
cur.fetchone()

# Timed lookups
latencies_ms = []
for txn_id in lookup_ids:
    t0 = time.perf_counter()
    cur.execute("SELECT * FROM fraud.txn_features WHERE txn_id = %s", (txn_id,))
    cur.fetchone()
    latencies_ms.append((time.perf_counter() - t0) * 1000)

cur.close()
conn.close()

latencies_ms.sort()
p50 = latencies_ms[len(latencies_ms) // 2]
p95 = latencies_ms[int(len(latencies_ms) * 0.95)]
p99 = latencies_ms[int(len(latencies_ms) * 0.99)]

print(f"Read latency (200 PK lookups on reused connection, after warm-up):")
print(f"  p50: {p50:.2f} ms")
print(f"  p95: {p95:.2f} ms")
print(f"  p99: {p99:.2f} ms")
print(f"  min: {min(latencies_ms):.2f} ms, max: {max(latencies_ms):.2f} ms")

Read latency (200 PK lookups on reused connection, after warm-up):
  p50: 3.04 ms
  p95: 4.63 ms
  p99: 7.56 ms
  min: 2.39 ms, max: 8.03 ms


In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

for name in [
    "synced_tables/fe_bar_lakebase.fraud.txn_features",
    "synced_tables/fe_bar_lakebase.fraud.customer_risk_profile",
]:
    st = w.postgres.get_synced_table(name=name)
    short = name.split(".")[-1]
    print(f"\n=== {short} ===")
    print(f"  State: {st.status.detailed_state.value}")
    print(f"  Pipeline: {st.status.pipeline_id}")
    if st.status.last_sync_time:
        print(f"  Last sync: {st.status.last_sync_time}")
    if st.spec:
        print(f"  Source: {st.spec.source_table_full_name}")
        print(f"  PK: {st.spec.primary_key_columns}")
        print(f"  Mode: {st.spec.scheduling_policy}")
        print(f"  PG database: {st.spec.postgres_database}")


=== txn_features ===
  State: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
  Pipeline: f8690359-4bb1-45b9-8fdb-5bd892239db2

=== customer_risk_profile ===
  State: SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE
  Pipeline: f8690359-4bb1-45b9-8fdb-5bd892239db2
